# Israeli legal eval on Kaggle (T4 x2): Gemma 3 27B (4-bit QAT) with the 29 Sept fixes

The same test as the 29 Sept Qwen 14B / Gemma 12B run: **the same 100 questions** (stratified sample,
seed 2026) and **the first 5 paralegal cases**, with retrieval over the bulk legal corpus. This time
it's only **`gemma3:27b-it-qat`** (Gemma 3 27B, multimodal, 4-bit quantization-aware; no thinking mode),
on the code with the fixes recommended in the 29 Sept report:

| Fix | Where | What the 12B run showed |
|---|---|---|
| Out-of-scope records never retrieved: West Bank military orders, the Jordanian criminal law, drafts; repealed law only when the plan names it with its year | `legal/corpus_retrieval.in_scope`, `config legal.corpus` | 15-18 questions pulled West Bank orders, 9-11 the Jordanian law, 9-10 the repealed 1984 Civil Procedure Regulations |
| BM25 over the corpus fused with the dense search | `legal/corpus_lexical.py` | the governing section was missing for 16 of 98 questions, which then scored 33% vs 76% |
| Plan budget 512 -> 1024 tokens, issues as short Hebrew phrases | `PLAN_MAX_TOKENS`, `PLAN_PROMPT` | 4 plans cut off mid-JSON |
| Embedder + reranker on the GPU (fp16) | `legal.retrieval.device` | CPU retrieval was ~74% of Gemma's wall time |
| כן/לא opener only when the question asks yes/no; a stray one is dropped | `ANSWER_SYSTEM`, `split_yes_no_opener` | 55 of 66 non-yes/no answers opened with כן/לא |
| A rule_conclusion label is checked against its own explanation | `check_label` | IL-016, IL-214, IL-219: the label contradicted the reasoning |
| Other-script rewrite: up to two passes | `MAX_REWRITE_PASSES` | (Qwen: 26 answers) |
| Cases: planned retrieval, stricter work-file rules, missing sections written, unsupported dates/amounts revised | `eval_cases.py` | cases averaged 42.8/100; every case lost points for an invented period, a law applied out of scope or a misread fact |

**No judging here**: send the results zip back to Claude, which judges it with the same rubric as the
12B run and compares. Step 6 prints a judge-free comparison against the 12B numbers right away
(multiple choice, yes/no labels, governing section retrieved, out-of-scope sources, openers, timing).

**Model vs. fixes.** This run changes both at once. To see how much the fixes alone moved things,
uncomment `gemma3:12b` in `MODELS` below: the 12B then runs on the same fixed code (about 1.5 hours more).

## Before running
1. **Add-ons -> Secrets**: `GITHUB_TOKEN` with read access to `zananiri/AI-IZ` (the same secret as before).
2. **Add Input**: the corpus dataset built by `notebooks/kaggle_legal_corpus_build.ipynb`
   (`legal_corpus_vectordb.zip` or its unzipped folder) -- the same one as the 12B run.
3. Settings -> Accelerator: **GPU T4 x2** -- required this time: the 27B model (~18 GB) is split over both GPUs.
4. Settings -> Internet: **on**.
5. **Run All**.

## Time
Roughly: loading the model and building the BM25 index ~10 min; ~1.5-2 min per question (100 questions:
~3 h); ~10 min per case (5 cases: ~1 h) -- **about 4-5 hours**, well inside Kaggle's 12-hour session.
The answers are resumable: see `RESUME_FROM`.

In [ ]:
BRANCH = "main"            # the branch to test
MODELS = [                 # answered in this order
    {"tag": "gemma3:27b-it-qat", "thinking": False},   # Gemma 3 27B, 4-bit QAT, multimodal; no thinking mode
    # {"tag": "gemma3:12b", "thinking": False},        # optional: the 12B on the same fixed code (fixes vs. model)
]
LIMIT = 100             # questions: a stratified sample across the 9 categories; None = all 500
SAMPLE_SEED = 2026      # the same seed as the other runs -> the same 100 questions
RUN_CASES = True        # the paralegal cases
CASE_LIMIT = 5          # the first N cases of cases_gold.json
TOP_K = 12              # retrieved chunks per question (issue spotting: 16, set in eval_run.py)
CASE_TOP_K = 16         # retrieved chunks per case
CONTEXT_LENGTH = 16384  # tokens per call (Ollama num_ctx): prompt + answer
ANSWER_MAX_TOKENS = 6144
CASE_MAX_TOKENS = 6144
REQUEST_TIMEOUT_S = 1800  # per model call
RETRIEVAL_DEVICE = "cuda"  # embedder + reranker; falls back to the CPU by itself if the GPUs are full

RUN_NAME = "corpus_gemma27b_fixes"   # RUN_DIR (data/legal/eval_gemma27b/RUN_NAME) is set in step 1
RESUME_FROM = None      # a path (or "auto") to an earlier run's results zip/folder attached as Input; None = fresh

# The 29 Sept run (gemma3:12b, before the fixes; evals/corpus_qwen14b_gemma12b_judged/ in the repo checkout
# on the PC), for the comparison in step 6.
BASELINE = {"model": "gemma3:12b (29 Sept, before the fixes)",
            "multiple_choice": 0.812, "yes_no_label": 0.722, "governing_section_retrieved_pct": 83.7,
            "out_of_scope_questions": {"West Bank military orders": 15, "Jordanian criminal law": 11,
                                       "Repealed Civil Procedure Regs 1984": 9, "Herut draft constitution": 4},
            "non_yes_no_answers_opening_with_label": "55 of 66", "plans_cut_off": 4,
            "median_answer_s": 9.8, "median_plan_s": 9.1,
            "judged_overall": 69.2, "cases_average": 42.8}

## 1. Code, packages and the unit tests

In [ ]:
import os, pathlib, shutil, subprocess, json, time, urllib.request
from kaggle_secrets import UserSecretsClient

def sh(command, log=None, check=True, env=None):
    """Runs a shell command with its output streamed live (and appended to `log`); raises on
    failure when check=True -- unlike `!command`, which carries on silently."""
    proc = subprocess.Popen(["bash", "-o", "pipefail", "-c", command], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    out = open(log, "a", encoding="utf-8") if log else None
    for line in proc.stdout:
        print(line, end="", flush=True)
        if out:
            out.write(line)
    if out:
        out.close()
    if proc.wait() and check:
        raise RuntimeError(f"exit code {proc.returncode}: {command}")
    return proc.returncode

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
assert len(gpus) >= 2, "Settings -> Accelerator: GPU T4 x2 -- the 27B model needs both GPUs"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "/kaggle/working/AI-IZ"
if not pathlib.Path(REPO, ".git").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    f"https://{token}@github.com/zananiri/AI-IZ.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/zananiri/AI-IZ.git"], check=True)
os.chdir(REPO)
RUN_DIR = pathlib.Path(f"data/legal/eval_gemma27b/{RUN_NAME}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
sh("git log --oneline -1")

sh('pip install -q uv')
sh(f'uv pip install -q --system -e ".[legal,legal-data]" pytest')

# The code this run depends on, tested first (a failure is printed, not fatal; the full suite has a few
# known failures unrelated to the Legal eval).
sh("python -m pytest -q -p no:cacheprovider tests/unit/test_legal_corpus_lexical.py "
   "tests/unit/test_legal_corpus_retrieval.py tests/unit/test_legal_eval_repairs.py tests/unit/test_llm_client.py",
   log=RUN_DIR / "unit_tests.log", check=False)

## 2. Ollama and the model

`OLLAMA_MAX_LOADED_MODELS=1`: one model at a time. Flash attention + an 8-bit KV cache halve the 16k context's
memory. `OLLAMA_KEEP_ALIVE=-1`: the model stays loaded between the scripts, so it isn't reloaded after the
retrieval models have taken their share of a GPU. The 27B (~18 GB) doesn't fit one T4 (15 GB), so Ollama
splits it over both.

In [ ]:
sh("apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null")
if not shutil.which("ollama"):
    sh("curl -fsSL https://ollama.com/install.sh | sh > /dev/null")

OLLAMA_LOG = RUN_DIR / "ollama.log"
if subprocess.run(["curl", "-sf", "http://localhost:11434/api/tags"], capture_output=True).returncode:
    ollama = subprocess.Popen(["ollama", "serve"], stdout=open(OLLAMA_LOG, "a"), stderr=subprocess.STDOUT,
                              env={**os.environ, "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_FLASH_ATTENTION": "1",
                                   "OLLAMA_KV_CACHE_TYPE": "q8_0", "OLLAMA_KEEP_ALIVE": "-1"})
    for _ in range(60):
        if not subprocess.run(["curl", "-sf", "http://localhost:11434/api/tags"], capture_output=True).returncode:
            break
        time.sleep(2)

for m in MODELS:
    sh(f"ollama pull {m['tag']}")
sh("ollama list")

## 3. The bulk legal corpus (attached input), its BM25 index, and the eval set (from the repo)

The corpus comes from the dataset attached in step 1. BM25 reads each category's `lexical_<category>.jsonl`
(written by `vectorize.py --export-lexical`, part of the corpus build); if a corpus has only the state
database, it's exported here. The index is built once (a few minutes) and cached beside it. Also checks every
law and section the gold answers cite against the installed corpus (`corpus_coverage.md`).

In [ ]:
input_zips = sorted(pathlib.Path("/kaggle/input").rglob("legal_corpus_vectordb*.zip"))
input_dirs = [p for p in pathlib.Path("/kaggle/input").rglob("*")
              if p.is_dir() and any((p / c).is_dir() for c in ("laws", "procedural_rules"))]
assert input_zips or input_dirs, "attach the corpus dataset built by notebooks/kaggle_legal_corpus_build.ipynb (Add Input)"
VECTORDB = pathlib.Path("data/legal_corpus_vectordb")
if input_zips:
    sh(f'python scripts/legal_data/install_corpus.py "{input_zips[0]}" --replace')
else:
    shutil.rmtree(VECTORDB, ignore_errors=True)
    shutil.copytree(input_dirs[0], VECTORDB)
    print("copied", input_dirs[0], "->", VECTORDB)

CORPUS_CATEGORIES = sorted(p.name for p in VECTORDB.iterdir() if p.is_dir() and (p / "chroma.sqlite3").exists())
print("corpus categories:", CORPUS_CATEGORIES)
missing = [c for c in ("laws", "procedural_rules") if c not in CORPUS_CATEGORIES]
if missing:
    print(f"[warn] the corpus has no {missing}: questions on those will retrieve nothing from them")

no_lexical = [c for c in CORPUS_CATEGORIES if not (VECTORDB / c / f"lexical_{c}.jsonl").exists()]
if no_lexical:
    sh(f'python scripts/legal_data/vectorize.py --vectordb {VECTORDB} --categories {",".join(no_lexical)} --export-lexical',
       check=False)
started = time.time()
sh("""python - <<'EOF'
from docslides.config import get_config
from docslides.legal.corpus_lexical import lexical_index
cfg = get_config().legal.corpus
for category in cfg.categories:
    index = lexical_index(cfg.vectordb_dir, category)
    print(category, "BM25:", "off (no lexical file)" if index is None else f"{len(index.ids)} chunks, {len(index.vocabulary)} terms")
EOF""")
print(f"BM25 ready in {time.time() - started:.0f}s")

EVAL_DIR = pathlib.Path("legal_txt/Evals")
import zipfile
zipfile.ZipFile(EVAL_DIR / "israeli_legal_eval.zip").extractall(EVAL_DIR)
EVAL = EVAL_DIR / "israeli_legal_eval"
assert (EVAL / "questions.jsonl").exists(), f"eval set not found under {EVAL}"

sh(f'python scripts/legal_data/check_corpus_coverage.py --gold {EVAL / "gold.jsonl"} --categories {",".join(CORPUS_CATEGORIES)} '
   f'--out "{RUN_DIR / "corpus_coverage.md"}" --json-out "{RUN_DIR / "corpus_coverage.json"}"', check=False)

## 4. The same 100 questions as the other runs

A proportional slice of each of the 9 categories, picked with `SAMPLE_SEED` -- the same ids as the 12B run.

In [ ]:
import random
from collections import Counter, defaultdict

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

all_questions = load_jsonl(EVAL / "questions.jsonl")
QUESTIONS = RUN_DIR / "questions_sample.jsonl"
if LIMIT and LIMIT < len(all_questions):
    by_cat = defaultdict(list)
    for q in all_questions:
        by_cat[q["category"]].append(q)
    share, rng, sample = LIMIT / len(all_questions), random.Random(SAMPLE_SEED), []
    for qs in by_cat.values():
        k = max(1, round(len(qs) * share))
        picked = qs[:k] if SAMPLE_SEED is None else rng.sample(qs, k)
        sample += sorted(picked, key=lambda q: q["id"])
else:
    sample = all_questions
with open(QUESTIONS, "w", encoding="utf-8") as f:
    for q in sample:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")
print(f"{len(sample)} questions ->", QUESTIONS)
print("by category:", dict(Counter(q["category"] for q in sample).most_common()))

## 4b. Resume: bring in an earlier run's results (if `RESUME_FROM` is set)

Copies the earlier `answers_*` files into place before anything runs, minus answers that came back empty or
errored -- so those get redone instead of being skipped as "already done".

In [ ]:
def model_dir(tag):
    return RUN_DIR / tag.replace(":", "_").replace("/", "_")

def find_eval_dir(root):
    first = model_dir(MODELS[0]["tag"]).name
    hits = sorted(p for p in pathlib.Path(root).rglob(RUN_NAME) if p.is_dir() and (p / first).exists())
    return hits[0] if hits else None

if RESUME_FROM:
    src = RESUME_FROM
    if src == "auto":
        zips = sorted(pathlib.Path("/kaggle/input").rglob(f"*{RUN_NAME}*results*.zip"))
        assert len(zips) <= 1, f"several earlier results attached, remove all but one from Input: {zips}"
        src = zips[0] if zips else "/kaggle/input"
    src = pathlib.Path(src)
    if src.is_file() and src.suffix == ".zip":
        extract_to = pathlib.Path("/kaggle/working/resume_from")
        shutil.rmtree(extract_to, ignore_errors=True)
        zipfile.ZipFile(src).extractall(extract_to)
        src = extract_to
    prev = find_eval_dir(src)
    assert prev, f"no {RUN_NAME}/<model> folders found under {src}"
    for m in MODELS:
        prev_dir, out_dir = prev / model_dir(m["tag"]).name, model_dir(m["tag"])
        for rel in ("bulk500/answers_rag.jsonl", "cases/answers_rag.jsonl"):
            src_file = prev_dir / rel
            if src_file.exists():
                (out_dir / rel).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(src_file, out_dir / rel)
        prev_trace, out_trace = prev_dir / "llm_trace", out_dir / "llm_trace"
        if prev_trace.exists():
            out_trace.mkdir(parents=True, exist_ok=True)
            for f in prev_trace.glob("*.jsonl"):
                shutil.copy(f, out_trace / f.name)
    print("resumed from", prev)

## 5. Answer (resumable)

Each model gets its own config (Ollama, its tag, thinking off, the 16k context, retrieval on the GPU) and
its own output folder and trace. The model is loaded into Ollama **first**, at the same context size the
scripts ask for, so it gets the GPU memory it needs; the embedder and reranker then go on whichever GPU has
more room left (or the CPU, if neither has -- the log line `retrieval ready on ...` says which). Each line
of output is `id (category): Ns`.

In [ ]:
import yaml

base_cfg = yaml.safe_load(open("config/config.yaml", encoding="utf-8"))

def model_env(m):
    out = model_dir(m["tag"])
    for sub in ("bulk500", "cases", "llm_trace"):
        (out / sub).mkdir(parents=True, exist_ok=True)
    cfg = json.loads(json.dumps(base_cfg))
    orch = cfg["legal"]["orchestrator"]
    orch.update(backend="ollama", base_url="http://localhost:11434", model=m["tag"],
                max_model_len=CONTEXT_LENGTH, request_timeout_s=REQUEST_TIMEOUT_S, supports_thinking=m["thinking"])
    cfg["legal"].setdefault("retrieval", {})["device"] = RETRIEVAL_DEVICE
    cfg.setdefault("logging", {})["llm_trace_dir"] = str(out / "llm_trace")
    cfg_path = out / "config.yaml"
    cfg_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
    return out, {**os.environ, "DOCSLIDES_CONFIG": str(cfg_path),
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_BACKEND": "ollama",
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_BASE_URL": "http://localhost:11434",
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_MODEL": m["tag"],
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_MAX_MODEL_LEN": str(CONTEXT_LENGTH),
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_SUPPORTS_THINKING": str(m["thinking"]).lower(),
                 "DOCSLIDES_LLM_TRACE_DIR": str(out / "llm_trace")}

def load_model(tag):
    """Loads `tag` into Ollama at CONTEXT_LENGTH (the num_ctx the scripts send, so their first call doesn't
    reload it) and shows how it was placed."""
    body = json.dumps({"model": tag, "prompt": "", "keep_alive": -1, "options": {"num_ctx": CONTEXT_LENGTH}}).encode()
    started = time.time()
    urllib.request.urlopen(urllib.request.Request("http://localhost:11434/api/generate", data=body,
                                                  headers={"Content-Type": "application/json"}), timeout=1800).read()
    print(f"{tag} loaded in {time.time() - started:.0f}s")
    sh("ollama ps; nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv")

def drop_failed(path):
    """Resume: remove answers that errored or came back empty, so this run redoes them."""
    if not path.exists():
        return 0
    rows = load_jsonl(path)
    good = [r for r in rows if not r.get("error") and str(r.get("answer") or r.get("work_file") or "").strip()]
    if len(good) < len(rows):
        with open(path, "w", encoding="utf-8") as f:
            for r in good:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return len(rows) - len(good)

for m in MODELS:
    out, env = model_env(m)
    log = out / "run.log"
    no_think = "" if m["thinking"] else " --no-thinking"
    load_model(m["tag"])
    answers = out / "bulk500" / "answers_rag.jsonl"
    print(f"\n{'=' * 20} {m['tag']}: questions {'=' * 20}  (redoing {drop_failed(answers)} failed)")
    sh(f'python scripts/legal_data/eval_run.py answer --questions "{QUESTIONS}" --top-k {TOP_K} '
       f'--max-tokens {ANSWER_MAX_TOKENS} --out "{answers}"{no_think}', log=log, env=env)
    if RUN_CASES:
        cases = out / "cases" / "answers_rag.jsonl"
        print(f"\n{'=' * 20} {m['tag']}: {CASE_LIMIT} cases {'=' * 20}  (redoing {drop_failed(cases)} failed)")
        sh(f'python scripts/legal_data/eval_cases.py answer --dir {EVAL} --top-k {CASE_TOP_K} '
           f'--max-tokens {CASE_MAX_TOKENS} --limit {CASE_LIMIT} --out "{cases}"{no_think}', log=log, env=env)
    subprocess.run(["ollama", "stop", m["tag"]], capture_output=True)  # free the GPUs for the next model

## 6. Reasoning logs, judge-free scores, and the comparison with the 12B run

- `reasoning_report.md`: per question/case, every call's output, stop reason and timing;
  `reasoning_report_with_prompts.md` adds the full prompts (retrieved evidence included).
- `judge_free_report.*`: score.py (unmodified) without a judge -- the 16 multiple-choice and 18 yes/no
  label scores only; the judged part waits for review.
- `comparison.md`: this run next to the 12B baseline on everything that needs no judge.

In [ ]:
import re, statistics

FOREIGN = re.compile(r"[\u0400-\u04FF\u0600-\u06FF\u4E00-\u9FFF\u3040-\u30FF\uAC00-\uD7A3\u0E00-\u0E7F]+|[A-Za-z]{4,}")
# The same matching as the 29 Sept analysis, so the numbers compare.
NIQQUD = re.compile(r"[\u0591-\u05C7]")
YEAR = re.compile(r"(?:^|\s)ה?תש[א-ת]{1,3}(?=\s|$)|\d{4}")
SECTION = re.compile(r"\d{1,4}[א-ת]{0,3}")
OUT_OF_SCOPE = {"West Bank military orders": re.compile(r"\(יהודה והשומרון\)"),
                "Jordanian criminal law": re.compile(r"החוק הפלילי הירדני"),
                "Repealed Civil Procedure Regs 1984": re.compile(r"תקנות סדר הדין האזרחי, התשמ״ד–1984"),
                "Herut draft constitution": re.compile(r"הצעת תנועת החירות")}
YES_NO_OPENER = re.compile(r"^\s*(?:\*\*)?\s*(כן|לא)\s*(?:\*\*)?\s*(?:[,.:;!\-–—]+|\n)")

def law_key(name):
    name = NIQQUD.sub("", (name or "").replace("־", " ").split(",")[0])
    name = re.sub(r"[\"'״׳`()\[\]\-–—\s]+", " ", name)
    name = YEAR.sub(" ", name)
    return " ".join(w[1:] if w.startswith("ה") and len(w) > 3 else w for w in name.split())

def same_law(a, b):
    a, b = law_key(a), law_key(b)
    return bool(a) and bool(b) and (a == b or a in b or b in a)

def base_section(s):
    found = SECTION.search(str(s or ""))
    return found.group(0) if found else None

gold = {g["id"]: g for g in load_jsonl(EVAL / "gold.jsonl")}
questions = {q["id"]: q for q in sample}

def governing_section_retrieved(row):
    cites = [c for c in gold.get(row["id"], {}).get("citations", []) if c.get("law")]
    if not cites:
        return None
    for c in cites:
        sec = base_section(c.get("section"))
        for r in row.get("retrieved") or []:
            if same_law(c["law"], r.get("title") or "") and (sec is None or base_section(r.get("section")) == sec):
                return True
    return False

summary = {"run": RUN_NAME, "questions": len(sample), "cases": CASE_LIMIT if RUN_CASES else 0, "models": {}}
for m in MODELS:
    out = model_dir(m["tag"])
    traces = sorted((out / "llm_trace").glob("*.jsonl"))
    if traces:
        files = " ".join(f'"{p}"' for p in traces)
        sh(f'python scripts/llm_trace_report.py {files} > "{out / "reasoning_report.md"}"', check=False)
        sh(f'python scripts/llm_trace_report.py {files} --prompts > "{out / "reasoning_report_with_prompts.md"}"', check=False)
    answers = out / "bulk500" / "answers_rag.jsonl"
    judge_free = {}
    if answers.exists():
        sh(f'python {EVAL / "score.py"} report --gold {EVAL / "gold.jsonl"} --answers "{answers}" '
           f'--json-out "{out / "judge_free_report.json"}" > "{out / "judge_free_report.txt"}"', check=False)
        if (out / "judge_free_report.json").exists():
            rows = json.load(open(out / "judge_free_report.json", encoding="utf-8"))["rows"]
            mcq = [r["score"] for r in rows if r["category"] == "mcq_bar"]
            labels = [r["label_correct"] for r in rows if "label_correct" in r]
            judge_free = {"multiple_choice": round(statistics.mean(mcq), 3) if mcq else None, "mcq_n": len(mcq),
                          "yes_no_label": round(statistics.mean(labels), 3) if labels else None, "yes_no_n": len(labels)}
    ans = load_jsonl(answers) if answers.exists() else []
    case_rows = load_jsonl(out / "cases" / "answers_rag.jsonl") if (out / "cases" / "answers_rag.jsonl").exists() else []
    calls = [json.loads(l) for p in traces for l in open(p, encoding="utf-8") if l.strip()]
    by_site = defaultdict(list)
    for c in calls:
        by_site[c.get("call_site")].append(c)
    hits = [h for h in (governing_section_retrieved(a) for a in ans) if h is not None]
    repairs = Counter(r for a in ans for r in a.get("repairs", []))
    non_yes_no = [a for a in ans if questions.get(a["id"], {}).get("category") not in ("rule_conclusion", "mcq_bar")]
    summary["models"][m["tag"]] = {
        "answers": len(ans), "errors": sum(bool(a.get("error")) for a in ans),
        "empty": sum(not str(a.get("answer") or "").strip() for a in ans),
        "repairs": dict(repairs),
        "answers_with_other_script_words": sum(bool(FOREIGN.search(a.get("answer") or "")) for a in ans),
        "governing_section_retrieved_pct": round(100 * sum(hits) / len(hits), 1) if hits else None,
        "out_of_scope_questions": {name: sum(any(rx.search(r.get("title") or "") for r in a.get("retrieved") or [])
                                             for a in ans) for name, rx in OUT_OF_SCOPE.items()},
        # written by the model, before the repair dropped it (the answer file has it removed)
        "non_yes_no_answers_opening_with_label": f"{repairs.get('stray_yes_no_opener', 0) + sum(bool(YES_NO_OPENER.match(a.get('answer') or '')) for a in non_yes_no)} of {len(non_yes_no)}",
        "empty_plans": sum(not a.get("plan") for a in ans if not a.get("error")),
        "cases": len(case_rows), "case_errors": sum(bool(c.get("error")) for c in case_rows),
        "case_repairs": dict(Counter(r for c in case_rows for r in c.get("repairs", []))),
        "judge_free": judge_free,
        "calls": {site: {"n": len(cs), "stopped_at_length": sum(c.get("done_reason") == "length" for c in cs),
                         "errors": sum(bool(c.get("error")) for c in cs),
                         "median_s": round(statistics.median([c.get("seconds") or 0 for c in cs]), 1),
                         "max_prompt_tokens": max((c.get("prompt_tokens") or 0) for c in cs),
                         "max_completion_tokens": max((c.get("completion_tokens") or 0) for c in cs)}
                  for site, cs in sorted(by_site.items())},
    }

(RUN_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")
lines = [f"# Kaggle eval run {RUN_NAME}", "", f"{summary['questions']} questions, {summary['cases']} cases", ""]
for tag, s in summary["models"].items():
    jf, calls = s["judge_free"], s["calls"]
    lines += [f"## {tag}", "",
              f"- answers {s['answers']} (errors {s['errors']}, empty {s['empty']}), cases {s['cases']} (errors {s['case_errors']})",
              f"- judge-free: multiple choice {jf.get('multiple_choice')} (n={jf.get('mcq_n')}), yes/no label {jf.get('yes_no_label')} (n={jf.get('yes_no_n')})",
              f"- repairs {s['repairs']}; case repairs {s['case_repairs']}; answers with other-script words: {s['answers_with_other_script_words']}", "",
              "| call site | calls | stopped at length | errors | median s | max prompt tok | max output tok |",
              "|---|---|---|---|---|---|---|"]
    lines += [f"| {site} | {c['n']} | {c['stopped_at_length']} | {c['errors']} | {c['median_s']} | {c['max_prompt_tokens']} | {c['max_completion_tokens']} |"
              for site, c in calls.items()] + [""]
    lines += ["### Against the 12B baseline (judge-free)", "", f"| | {BASELINE['model']} | {tag} |", "|---|---|---|",
              f"| multiple choice | {BASELINE['multiple_choice']} | {jf.get('multiple_choice')} |",
              f"| yes/no label | {BASELINE['yes_no_label']} | {jf.get('yes_no_label')} |",
              f"| governing section retrieved (%) | {BASELINE['governing_section_retrieved_pct']} | {s['governing_section_retrieved_pct']} |"]
    lines += [f"| questions retrieving {name} | {BASELINE['out_of_scope_questions'][name]} | {n} |"
              for name, n in s["out_of_scope_questions"].items()]
    lines += [f"| non-yes/no answers opening with כן/לא (as written) | {BASELINE['non_yes_no_answers_opening_with_label']} | {s['non_yes_no_answers_opening_with_label']} |",
              f"| plans cut off | {BASELINE['plans_cut_off']} | {calls.get('legal_eval_plan', {}).get('stopped_at_length')} |",
              f"| median answer call (s) | {BASELINE['median_answer_s']} | {calls.get('legal_eval_baseline', {}).get('median_s')} |",
              f"| median plan call (s) | {BASELINE['median_plan_s']} | {calls.get('legal_eval_plan', {}).get('median_s')} |",
              f"| judged overall / cases average | {BASELINE['judged_overall']} / {BASELINE['cases_average']} | after Claude's judging |", ""]
(RUN_DIR / "summary.md").write_text("\n".join(lines), encoding="utf-8")
(RUN_DIR / "comparison.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

## 7. Environment record and the results zip (for Output)

In [ ]:
env_lines = []
for label, cmd in [("git", "git log -1 --format='%H %s'"), ("gpu", "nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv"),
                   ("ollama", "ollama --version"), ("models", "ollama list"),
                   ("python", "python -c \"import sys, chromadb, sentence_transformers, torch, sklearn; print(sys.version); print('chromadb', chromadb.__version__, '| sentence-transformers', sentence_transformers.__version__, '| torch', torch.__version__, '| scikit-learn', sklearn.__version__)\"")]:
    res = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)
    env_lines += [f"## {label}", (res.stdout + res.stderr).strip(), ""]
env_lines += ["## settings", json.dumps({k: globals()[k] for k in ("BRANCH", "MODELS", "LIMIT", "SAMPLE_SEED", "RUN_CASES", "CASE_LIMIT",
              "TOP_K", "CASE_TOP_K", "CONTEXT_LENGTH", "ANSWER_MAX_TOKENS", "CASE_MAX_TOKENS", "REQUEST_TIMEOUT_S",
              "RETRIEVAL_DEVICE")}, indent=1)]
(RUN_DIR / "environment.txt").write_text("\n".join(env_lines), encoding="utf-8")

DEST = f"/kaggle/working/{RUN_NAME}_results.zip"
sh(f'cd /kaggle/working && zip -qr {DEST} AI-IZ/{RUN_DIR}')
print("results zip:", DEST, f"({pathlib.Path(DEST).stat().st_size / 1e6:.1f} MB)")
print("Send this zip back for the judging and the comparison with the 12B run.")